In [7]:
import re
import pandas as pd
from Levenshtein import distance
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter

# Chemins des fichiers
chemin_metivier = r"C:\\Users\\louis\\OneDrive - Université de Poitiers\\Fac\\Master\\Mémoire\\Variation flexions\\Textes locuteurs\\Poitevin\\Formes trouvées\\Adolphe Métivier.txt"
chemin_radicaux = r"C:\\Users\\louis\\OneDrive - Université de Poitiers\\Fac\\Master\\Mémoire\\Variation flexions\\Lexiques\\Lexiques radicaux infinitifs\\Infinitifs.tsv"
chemin_sortie = r"C:\\Users\\louis\\OneDrive - Université de Poitiers\\Fac\\Master\\Mémoire\\Variation flexions\\Dicopoitevin\\metivier_dicopoitevin_levenshtein.xlsx"

# 1) Lire le fichier Métivier et récupérer les terminaisons + formes
with open(chemin_metivier, "r", encoding="cp1252") as f:
    lignes = f.readlines()

premiere_ligne = lignes[0].strip()
m = re.search(r"Inf\s*:\s*(.+)", premiere_ligne, flags=re.IGNORECASE)
if not m:
    raise ValueError("Impossible de trouver 'Term :' dans la première ligne du fichier Métivier.")

term_str = m.group(1)
terminaisons = [t.strip() for t in re.split(r"[,\\s]+", term_str) if t.strip()]
terminaisons = sorted(terminaisons, key=len, reverse=True)

# 2) Extraire les formes verbales
formes_metivier_brutes = [l.strip() for l in lignes[1:] if l.strip()]

def enlever_termination(mot, terminaisons):
    mot = mot.lower()
    for term in terminaisons:
        if mot.endswith(term) and len(mot) > len(term):
            return mot[: -len(term)]
    return mot

radicaux_metivier = [enlever_termination(m, terminaisons) for m in formes_metivier_brutes]

# 2bis) Garder un seul exemplaire de chaque radical Métivier
radicaux_metivier_uniques = {}
for mot_brut, rad in zip(formes_metivier_brutes, radicaux_metivier):
    if rad not in radicaux_metivier_uniques:
        radicaux_metivier_uniques[rad] = mot_brut

# Listes synchronisées sans doublons
radicaux_metivier_liste = list(radicaux_metivier_uniques.keys())
formes_metivier_liste = list(radicaux_metivier_uniques.values())

# 2ter) Ne garder que les radicaux de longueur >= 3
radicaux_filtrés = []
formes_filtrées = []
for rad, mot in zip(radicaux_metivier_liste, formes_metivier_liste):
    if len(rad) >= 3:
        radicaux_filtrés.append(rad)
        formes_filtrées.append(mot)

# 3) Lire le TSV infinitifs fr
df_fr = pd.read_csv(chemin_radicaux, sep="\t", encoding="utf-8")
if "radical" not in df_fr.columns:
    raise ValueError("La colonne 'radical' n'existe pas.")

radicaux_fr = df_fr["radical"].astype(str).str.lower().tolist()

# 4) Distance de Levenshtein normalisée (entre 0 et 1)

def levenshtein_norm(a, b):
    d = distance(a, b)
    max_len = max(len(a), len(b))
    if max_len == 0:
        return 0.0
    # normalisation + arrondi à 3 décimales
    return round(d / max_len, 3)

# Fonction utilitaire : vérifier que la première lettre est identique
# et qu'aucune insertion/suppression ne se produit au tout début.
def premiere_lettre_compatible(a, b):
    if not a or not b:
        return False
    # Si les premières lettres sont différentes, on disqualifie
    if a[0] != b[0]:
        return False
    # Cas insertion/suppression initiale à éviter, ex. 'accompagn' vs 'raccompagn'
    # Si l'une commence par l'autre première lettre + un caractère différent, on rejette.
    # Exemple : 'a...' et 'ra...' => première lettre différente => déjà géré.
    return True

# seuil de distance normalisée (à ajuster)
SEUIL_NORM = 0.3

resultats = []
for mot_brut, rad_met in zip(formes_filtrées, radicaux_filtrés):
    for rad_fr in radicaux_fr:
        # On écarte directement si la première lettre ne convient pas
        if not premiere_lettre_compatible(rad_met, rad_fr):
            continue

        d_norm = levenshtein_norm(rad_met, rad_fr)
        if d_norm <= SEUIL_NORM:
            resultats.append(
                {
                    "forme_metivier": mot_brut,
                    "radical_metivier": rad_met,
                    "radical_fr": rad_fr,
                    "distance_levenshtein_normalisee": d_norm,
                }
            )

df_res = pd.DataFrame(resultats)

# Sécurité : filtrer encore sur distance normalisée <= SEUIL_NORM
df_res = df_res[df_res["distance_levenshtein_normalisee"] <= SEUIL_NORM]

# s'assurer que la colonne est bien à 3 décimales
df_res["distance_levenshtein_normalisee"] = df_res["distance_levenshtein_normalisee"].round(3)

# 5) Sauvegarde en Excel
df_res.to_excel(chemin_sortie, index=False, sheet_name="Résultats")

# 6) Ajouter le filtre sur la ligne d'en-tête
wb = load_workbook(chemin_sortie)
ws = wb.active

max_col = get_column_letter(len(df_res.columns))
ws.auto_filter.ref = f"A1:{max_col}1"

ws.column_dimensions['A'].width = 25
ws.column_dimensions['B'].width = 20
ws.column_dimensions['C'].width = 25
ws.column_dimensions['D'].width = 20

wb.save(chemin_sortie)
print("Fichier Excel (radicaux >= 3 lettres, uniques, distance normalisée, filtres) créé :", chemin_sortie)


Fichier Excel (radicaux >= 3 lettres, uniques, distance normalisée, filtres) créé : C:\\Users\\louis\\OneDrive - Université de Poitiers\\Fac\\Master\\Mémoire\\Variation flexions\\Dicopoitevin\\metivier_dicopoitevin_levenshtein.xlsx
